In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from cuml.preprocessing import StandardScaler 
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/lung-cancer-dataset/dataset_med.csv


In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
import cudf as cd
from cuml.ensemble import RandomForestClassifier
from cuml.metrics import accuracy_score, confusion_matrix
from cuml.model_selection import train_test_split

In [3]:
data = pd.read_csv('/kaggle/input/lung-cancer-dataset/dataset_med.csv')
data

,id,age,gender,country,diagnosis_date,cancer_stage,family_history,smoking_status,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,survived
0,1,64.0,Male,Sweden,2016-04-05,Stage I,Yes,Passive Smoker,29.4,199,0,0,1,0,Chemotherapy,2017-09-10,0
1,2,50.0,Female,Netherlands,2023-04-20,Stage III,Yes,Passive Smoker,41.2,280,1,1,0,0,Surgery,2024-06-17,1
2,3,65.0,Female,Hungary,2023-04-05,Stage III,Yes,Former Smoker,44.0,268,1,1,0,0,Combined,2024-04-09,0
3,4,51.0,Female,Belgium,2016-02-05,Stage I,No,Passive Smoker,43.0,241,1,1,0,0,Chemotherapy,2017-04-23,0
4,5,37.0,Male,Luxembourg,2023-11-29,Stage I,No,Passive Smoker,19.7,178,0,0,0,0,Combined,2025-01-08,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889995,889996,40.0,Male,Malta,2022-07-01,Stage IV,No,Passive Smoker,44.8,243,1,1,1,0,Radiation,2023-02-23,0
889996,889997,62.0,Female,Cyprus,2015-09-27,Stage II,Yes,Former Smoker,21.6,240,0,0,0,0,Surgery,2017-06-19,0
889997,889998,48.0,Female,Estonia,2016-03-27,Stage III,Yes,Never Smoked,38.6,242,1,0,0,0,Combined,2017-01-23,1
889998,889999,67.0,Female,Slovakia,2015-12-22,Stage IV,Yes,Former Smoker,18.6,194,1,1,0,0,Chemotherapy,2017-12-12,0


In [4]:
data_id = data['id']
data = data.drop(columns=['id'], axis=1)

In [5]:
data['age'].unique()
data['age'].astype(int)
label_usia = ['child' ,'teen', 'young adult', 'established adult', 'pra-retirement', 'retiree', 'late elderly']
variabelusia = 'age'
bins = [0, 15, 25, 35, 55, 65, 75, data['age'].max()]
data['age_binning'] = pd.cut(
    data['age'], 
    bins=bins,
    labels=label_usia,
    right=False,
    include_lowest=True,
    duplicates='drop'
)
data['age_binning'] = data['age_binning'].astype('category').cat.codes
data = data.dropna()
data.describe()

,age,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,survived,age_binning
count,890000.000000,890000.000000,890000.000000,890000.000000,890000.000000,890000.000000,890000.000000,890000.000000,890000.000000
mean,55.007008,30.494172,233.633916,0.750024,0.469740,0.225956,0.088157,0.220229,3.695033
std,9.994485,8.368539,43.432278,0.432999,0.499084,0.418211,0.283524,0.414401,0.837311
min,4.000000,16.000000,150.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000
25%,48.000000,23.300000,196.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000
50%,55.000000,30.500000,242.000000,1.000000,0.000000,0.000000,0.000000,0.000000,4.000000
75%,62.000000,37.700000,271.000000,1.000000,1.000000,0.000000,0.000000,0.000000,4.000000
max,104.000000,45.000000,300.000000,1.000000,1.000000,1.000000,1.000000,1.000000,6.000000


In [6]:
data.duplicated().sum()
data.drop_duplicates(inplace=True)

In [7]:
data['gender'].unique()
data['gender'] = data['gender'].astype('category').cat.codes 

In [8]:
data['country'].unique() # one hot

array(['Sweden', 'Netherlands', 'Hungary', 'Belgium', 'Luxembourg',
       'Italy', 'Croatia', 'Denmark', 'Malta', 'Germany', 'Poland',
       'Ireland', 'Romania', 'Spain', 'Greece', 'Estonia', 'Cyprus',
       'France', 'Slovenia', 'Latvia', 'Portugal', 'Austria',
       'Czech Republic', 'Finland', 'Lithuania', 'Slovakia', 'Bulgaria'],
      dtype=object)

In [9]:
print(data['cancer_stage'])
data['cancer_stage'] = data['cancer_stage'].astype('category').cat.codes + 1
print(data['cancer_stage'])

0           Stage I
1         Stage III
2         Stage III
3           Stage I
4           Stage I
            ...    
889995     Stage IV
889996     Stage II
889997    Stage III
889998     Stage IV
889999     Stage II
Name: cancer_stage, Length: 890000, dtype: object
0         1
1         3
2         3
3         1
4         1
         ..
889995    4
889996    2
889997    3
889998    4
889999    2
Name: cancer_stage, Length: 890000, dtype: int8


In [10]:
data['family_history'].unique()
data['family_history'] = data['family_history'].map({'Yes': 1, 'No': 0})


In [11]:
data['smoking_status'].unique() # one hot

array(['Passive Smoker', 'Former Smoker', 'Never Smoked',
       'Current Smoker'], dtype=object)

In [12]:
data['treatment_type'].unique() # one hot

array(['Chemotherapy', 'Surgery', 'Combined', 'Radiation'], dtype=object)

In [13]:
print(data['diagnosis_date'])
print(data['end_treatment_date'])

data['diagnosis_date'] = pd.to_datetime(data['diagnosis_date'])
data['end_treatment_date'] = pd.to_datetime(data['end_treatment_date'])

data['time_gap'] = data['end_treatment_date'] - data['diagnosis_date']
data['treatment_day'] = data['time_gap'].dt.days
data = data.drop(columns=['time_gap', 'diagnosis_date', 'end_treatment_date'], axis=1)

0         2016-04-05
1         2023-04-20
2         2023-04-05
3         2016-02-05
4         2023-11-29
             ...    
889995    2022-07-01
889996    2015-09-27
889997    2016-03-27
889998    2015-12-22
889999    2021-07-26
Name: diagnosis_date, Length: 890000, dtype: object
0         2017-09-10
1         2024-06-17
2         2024-04-09
3         2017-04-23
4         2025-01-08
             ...    
889995    2023-02-23
889996    2017-06-19
889997    2017-01-23
889998    2017-12-12
889999    2022-10-19
Name: end_treatment_date, Length: 890000, dtype: object


In [14]:
data['age_stage'] = data['age'] * data['cancer_stage']

In [15]:
data = pd.get_dummies(data, columns=['country', 'smoking_status', 'treatment_type'])

In [16]:
data.info()
print(data.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 890000 entries, 0 to 889999
Data columns (total 49 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   age                            890000 non-null  float64
 1   gender                         890000 non-null  int8   
 2   cancer_stage                   890000 non-null  int8   
 3   family_history                 890000 non-null  int64  
 4   bmi                            890000 non-null  float64
 5   cholesterol_level              890000 non-null  int64  
 6   hypertension                   890000 non-null  int64  
 7   asthma                         890000 non-null  int64  
 8   cirrhosis                      890000 non-null  int64  
 9   other_cancer                   890000 non-null  int64  
 10  survived                       890000 non-null  int64  
 11  age_binning                    890000 non-null  int8   
 12  treatment_day                 

# Modeling

In [17]:
data = cd.DataFrame(data)

In [18]:
X = data.drop(columns= ['survived'], axis=1)
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_train, y_train, test_size=0.5, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.tranform(X_test)
X_val = scaler.transform(X_val)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_streams=1, split_criterion='gini')

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)
print(accurasy)

NameError: name 'X_val' is not defined